# Emojis classification

Baseline (see `main-cnn.ipynb`): a CNN trained from scratch on 72 px images reached 75.6% validation accuracy and 74.95% on the Kaggle test set. It overfitted (93% training accuracy) and its validation accuracy kept swinging between epochs, because its batch normalization layers behaved differently in training and evaluation mode.

`main-finetuning.ipynb` then fine-tuned a pretrained EfficientNetV2-B0 on 128 px images, with 5-fold cross-validation: 90.6% ± 1.2% cross-validation accuracy, 91.4% on Kaggle. Its errors are concentrated in a few vendor pairs: Facebook ↔ WhatsApp (31% of the errors), Apple/Google → Samsung (28%), Apple ↔ Google (9%) and Mozilla → Messenger (5%) (analysis in `IMPROVEMENT_PLAN.md`).

This notebook keeps that approach (transfer learning, white background, augmentation without flips, label smoothing, AdamW, 5 stratified folds averaged for the test predictions) and changes:
1. **Data fix**: the 409 duplicate training files created by the Drive upload (`xxxx (1).png`) are skipped. They were counted twice, and a copy could be validated by a model trained on its original.
2. **Higher resolution**: 224 px instead of 128 px, to keep the details of the 160 px (WhatsApp) and 240 px (Facebook, Messenger, Mozilla) source images.
3. **Stochastic depth restored**: EfficientNet's batch normalization is frozen layer by layer instead of with `training=False`, which also switched off its built-in stochastic depth (a regularization).
4. **Fixed training schedule**: one warmup epoch, then cosine decay, and the final weights are kept. Picking the best validation epoch made the cross-validation score optimistic.
5. **Test-time augmentation (TTA)**: each image is also predicted after small zooms and shifts, and the probabilities are averaged.
6. **Two backbones** (EfficientNetV2-B0 and ConvNeXt-Tiny), each run saved in its own folder, combined at the end into an **ensemble**, with a per-vendor bias correction.

The model only sees pixels: the original size and PNG mode of each image are loaded for error analysis only.

**How to run on Colab**
1. *Runtime → Change runtime type → GPU*. The data must be in `MyDrive/data` (`train/`, `test/`, `train_labels.csv`).
2. Choose `BACKBONE` in "Run configuration", then *Run all*.
3. If the session disconnects, *Run all* again: finished folds are loaded from Drive instead of retrained, and the preprocessed images are loaded from a cache.
4. "Ensemble of runs" combines every run saved in `MyDrive/emoji_outputs` and writes `submission.csv`.

## Imports
### Libraries

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, ops
from tensorflow.keras.applications import ConvNeXtTiny, EfficientNetV2B0
from tensorflow.keras.utils import to_categorical

In [ ]:
# On Colab, read the data from Google Drive and save outputs there, so they survive the session
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/data")
    OUTPUT_DIR = Path("/content/drive/MyDrive/emoji_outputs")
except ImportError:
    DATA_DIR = Path("data")
    OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs: {[tf.config.experimental.get_device_details(gpu).get('device_name') for gpu in gpus]}")
if gpus:
    # Compute in float16 on the GPU (about 2x faster on a T4), weights stay in float32
    tf.keras.mixed_precision.set_global_policy("mixed_float16")

### Run configuration

Two runs are planned. They use the same folds, so they can be combined (see "Ensemble of runs"):
- **Run A** (`main_run_a.ipynb`): `BACKBONE = "efficientnetv2b0"`, the previous model with the changes listed above, on all 5 folds (22 min per fold on a T4).
- **Run B** (this configuration): `BACKBONE = "convnext_tiny"`, a larger network with a different design. It takes about 6 min per epoch on a T4, 8× EfficientNet, so 5 folds would need about 9 hours of GPU, more than the free Colab quota. It is trained on fold 1 only (`FOLDS_TO_RUN = [0]` in "Cross-validation folds"): trained on the other 4 folds, validated on the same 1,976 images as run A's fold 1. Both networks are then compared on exactly the same images, as is their average (see "Ensemble of runs"). This decides whether ConvNeXt is worth more GPU time.

**Why 224 px:** Facebook, Messenger and Mozilla images come from 240 px sources and WhatsApp from 160 px. At 128 px they lose about half and a fifth of their resolution, and Facebook ↔ WhatsApp is the largest confusion. At 224 px they keep almost all their detail, and 224 px is the resolution both networks were pretrained at. It cannot help Apple, Google and Samsung (72 px sources, only enlarged), which only better features can separate: that is the motivation for ConvNeXt. Cost: about 3× the computation of 128 px.

In [ ]:
BACKBONE = "convnext_tiny"  # run B; "efficientnetv2b0" for run A
IMG_SIZE = 224

# Training settings of each backbone (see "Model training parameters")
BACKBONE_SETTINGS = {
    "efficientnetv2b0": {
        "network": EfficientNetV2B0, "finetune_epochs": 20, "peak_lr": 2e-4, "weight_decay": 0.004, "batch_size": 64,
    },
    # Batch size 32: at 64, ConvNeXt-Tiny at 224 px runs out of memory on a T4 GPU (16 GB)
    "convnext_tiny": {
        "network": ConvNeXtTiny, "finetune_epochs": 10, "peak_lr": 1e-4, "weight_decay": 0.05, "batch_size": 32,
    },
}
settings = BACKBONE_SETTINGS[BACKBONE]

# Each run saves its weights and predictions in its own folder, so runs can be resumed and combined
RUN_DIR = OUTPUT_DIR / f"{BACKBONE}_{IMG_SIZE}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run folder: {RUN_DIR}")

### Data

In [ ]:
def load_image(img_file: Path) -> np.ndarray:
    # Images mix RGBA and palette PNGs: composite onto a white background to get a uniform RGB array
    img = Image.open(img_file).convert("RGBA")
    background = Image.new("RGBA", img.size, (255, 255, 255, 255))
    img = Image.alpha_composite(background, img).convert("RGB")
    # Images come in several sizes (64 to 240 px, some non-square): pad to a white square, then resize
    side = max(img.size)
    square = Image.new("RGB", (side, side), (255, 255, 255))
    square.paste(img, ((side - img.width) // 2, (side - img.height) // 2))
    return np.array(square.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS))

def load_data(folder_path: Path):
    ids = []
    dataset = []
    # Original size and PNG mode of each file, e.g. "72 px P". For error analysis only: never given to the model
    sources = []
    n_copies = 0
    for img_file in sorted(folder_path.iterdir()):
        if img_file.suffix != ".png":
            continue
        # Re-uploading to Drive created copies named "xxxx (1).png": skip them so each image is counted once
        if " (" in img_file.stem:
            n_copies += 1
            continue
        ids.append(img_file.stem)
        dataset.append(load_image(img_file))
        with Image.open(img_file) as src:
            sources.append(f"{max(src.size)} px {src.mode}")
    print(f"{folder_path.name}: skipped {n_copies} duplicate copies")
    dataset_as_array = np.array(dataset)
    return dataset_as_array, ids, np.array(sources)

In [ ]:
# Decoding the ~20,000 PNG files from Drive takes several minutes: the preprocessed images are cached, one file
# per image size (about 3 GB at 224 px, delete it from Drive when the project is over)
cache_path = OUTPUT_DIR / f"images_{IMG_SIZE}px.npz"
if cache_path.exists():
    cache = np.load(cache_path)
    X_train, X_train_ids, X_train_sources = cache["X_train"], cache["X_train_ids"], cache["X_train_sources"]
    X_test, X_test_ids = cache["X_test"], cache["X_test_ids"]
    print(f"Loaded the preprocessed images from {cache_path}")
else:
    X_train, X_train_ids, X_train_sources = load_data(DATA_DIR / "train")
    X_test, X_test_ids, _ = load_data(DATA_DIR / "test")
    # Written under a temporary name, so an interrupted save never leaves a broken cache
    tmp_path = cache_path.with_suffix(".tmp.npz")
    np.savez(tmp_path, X_train=X_train, X_train_ids=X_train_ids, X_train_sources=X_train_sources,
             X_test=X_test, X_test_ids=X_test_ids)
    tmp_path.rename(cache_path)

labels = pd.read_csv(DATA_DIR / "train_labels.csv", dtype={"Id": str}).set_index("Id")
y_train = labels.loc[X_train_ids, "Label"].to_numpy()

# Each labelled image should appear exactly once
assert len(X_train_ids) == len(set(X_train_ids)) == len(labels), f"{len(X_train_ids)} images for {len(labels)} labels"

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
fig, axes = plt.subplots(2, 7, figsize=(14, 4))
for ax, img, label in zip(axes.flat, X_train, y_train):
    ax.imshow(img)
    ax.set_title(label)
    ax.axis("off")
plt.show()

### Preprocessing

Pixel values stay in [0, 255] (as `uint8`, which also saves memory): EfficientNetV2 rescales its inputs itself.

In [ ]:
# Encode string labels as integers, then one-hot encode
classes = np.array(sorted(set(y_train)))
num_classes = len(classes)
y_train_int = np.searchsorted(classes, y_train)
y_train_cat = to_categorical(y_train_int, num_classes=num_classes)

print(f"Classes: {list(classes)}")
print(f"Categorical y_train shape: {y_train_cat.shape}")

### Cross-validation folds

The training set is split into 5 folds. Each fold model trains on 4 folds and is validated on the remaining one, so every training image is used for validation exactly once. The folds are stratified: each keeps the vendor proportions of the full training set, which matters for the rare vendors (Mozilla has only 397 images).

In [ ]:
N_FOLDS = 5
# Run A used all folds: list(range(N_FOLDS)). Run B (ConvNeXt) only fold 1, to fit in the GPU budget
FOLDS_TO_RUN = [0]

rng = np.random.default_rng(0)
fold_of = np.empty(len(y_train_int), dtype=int)
for c in range(num_classes):
    class_idx = rng.permutation(np.flatnonzero(y_train_int == c))
    fold_of[class_idx] = np.arange(len(class_idx)) % N_FOLDS

pd.crosstab(fold_of, y_train, rownames=["fold"], colnames=["vendor"])

## Model
### Model architecture

- **Data augmentation** (training only): random shifts (±10%), zoom (±15%) and rotation (±11°). Empty areas are filled with white, like the emoji backgrounds (the baseline mirrored the image edges instead). There is no horizontal flip: vendor styles may have a consistent light direction (e.g. highlights on the upper left), which a flip would destroy.
- **A pretrained network** (ImageNet), without its classification layer. Its low-level features (edges, gradients, shading, textures) are what distinguishes vendor styles, and with about 10,000 training images, a pretrained network overfits much less than one trained from scratch. Both take pixel values in [0, 255] and rescale them themselves.
  - **EfficientNetV2-B0** (6M parameters), as in `main-finetuning.ipynb`.
  - **ConvNeXt-Tiny** (28M parameters): a larger, more recent design, that uses layer normalization instead of batch normalization. Being a different architecture, it should make different mistakes than EfficientNet, which is what an ensemble needs.
- Global average pooling (followed by layer normalization for ConvNeXt, as in its original classifier), dropout 30%, then a 7-neuron softmax layer.

**Batch normalization** (EfficientNet only): its layers are frozen, so they always use their stored ImageNet statistics, during training as in evaluation. This avoids the training/evaluation mismatch that made the baseline's validation accuracy unstable. `main-finetuning.ipynb` did this by calling the network with `training=False`, but that flag also switched off its **stochastic depth**: during training, EfficientNetV2 randomly skips the residual branch of its blocks for some images (up to 19% of the time in the deepest blocks), a regularization it was pretrained with. Freezing only the batch normalization layers keeps it, which matters for a model that reached 99.9% training accuracy against ~91% validation accuracy.

In [ ]:
def build_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # Augmentation layers are only active during training
    x = layers.RandomTranslation(0.1, 0.1, fill_mode="constant", fill_value=255.0)(inputs)
    x = layers.RandomZoom(0.15, fill_mode="constant", fill_value=255.0)(x)
    x = layers.RandomRotation(0.03, fill_mode="constant", fill_value=255.0)(x)

    base = settings["network"](include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(x)

    x = layers.GlobalAveragePooling2D()(x)
    if BACKBONE == "convnext_tiny":
        x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dropout(0.3)(x)
    # float32 output for numerical stability under mixed precision
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    return models.Model(inputs, outputs), base

def freeze_batch_norm(base):
    # A batch normalization layer that is not trainable uses its stored statistics, even during training.
    # Unlike base(x, training=False), this leaves the stochastic depth (dropout layers) of the base active
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False

model, base = build_model()
model.summary()

### Model training parameters

Each fold model is trained in two stages:
1. **Head only** (3 epochs): the pretrained base is frozen and only the new classification layer learns, so that its random initial weights don't send large, destructive gradients into the pretrained base.
2. **Fine-tuning** (20 epochs for EfficientNet, 10 for ConvNeXt, which fits the training set faster: 99% training accuracy after 4 epochs): the whole network trains with AdamW (Adam with weight decay). The learning rate rises linearly from 0 during the first epoch (warmup), so the first updates of the pretrained weights stay small while the optimizer's estimates of the gradient statistics are still unreliable, then decreases smoothly to 0 (cosine decay). Peak learning rate 2e-4 for EfficientNet (as before); 1e-4 and a stronger weight decay (0.05) for ConvNeXt, the usual settings for fine-tuning this larger network. Batches of 64 images (32 for ConvNeXt, which does not fit in the memory of a T4 GPU with 64 images of 224 px).

**Why not train longer:** in `main-finetuning.ipynb`, training accuracy passed 99% after 9 fine-tuning epochs, the training loss reached the minimum that label smoothing allows (0.446), and validation accuracy moved by less than 0.4% over the last epochs in every fold. More epochs would not have helped, so the schedule is shortened from 25 to 20 epochs, which pays for part of the higher resolution.

**Final weights, no early stopping:** the previous version kept the epoch with the best validation accuracy. Picking the best of ~25 noisy validation scores makes the validation accuracy optimistic (by 0.2% on average in the previous run, best vs last epoch), without gaining anything on the test set, since the last epochs only differ by noise. With a fixed schedule, the validation fold is only used to score the model, so the cross-validation accuracy is an unbiased estimate of the test accuracy.

The loss uses label smoothing (0.1): targets become 0.914 / 0.014 instead of 1 / 0, which discourages overconfident predictions and reduces overfitting.

In [ ]:
HEAD_EPOCHS = 3
WARMUP_EPOCHS = 1
FINETUNE_EPOCHS = settings["finetune_epochs"]  # including the warmup epoch
BATCH_SIZE = settings["batch_size"]

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

def train_fold(fold):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(fold)
    is_train = fold_of != fold
    X_tr, y_tr = X_train[is_train], y_train_cat[is_train]
    X_val, y_val = X_train[~is_train], y_train_cat[~is_train]

    model, base = build_model()

    # Stage 1: train only the classification head
    base.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3), loss=loss, metrics=["accuracy"])
    head_history = model.fit(
        X_tr, y_tr, validation_data=(X_val, y_val), epochs=HEAD_EPOCHS, batch_size=BATCH_SIZE
    )

    # Stage 2: fine-tune the whole network, except the batch normalization statistics
    base.trainable = True
    freeze_batch_norm(base)
    steps_per_epoch = int(np.ceil(len(X_tr) / BATCH_SIZE))
    schedule = tf.keras.optimizers.schedules.CosineDecay(
        0.0,
        decay_steps=(FINETUNE_EPOCHS - WARMUP_EPOCHS) * steps_per_epoch,
        warmup_target=settings["peak_lr"],
        warmup_steps=WARMUP_EPOCHS * steps_per_epoch,
    )
    optimizer = tf.keras.optimizers.AdamW(learning_rate=schedule, weight_decay=settings["weight_decay"])
    model.compile(optimizer=optimizer, loss=loss, metrics=["accuracy"])
    finetune_history = model.fit(
        X_tr, y_tr, validation_data=(X_val, y_val), epochs=FINETUNE_EPOCHS, batch_size=BATCH_SIZE
    )
    # Saved so the fold model can be reused (e.g. with metadata inputs) without retraining
    model.save_weights(RUN_DIR / f"fold{fold}.weights.h5")

    history = {
        key: [float(v) for v in head_history.history[key] + finetune_history.history[key]]
        for key in finetune_history.history
    }
    return model, history

### Test-time augmentation (TTA)

Each validation and test image is also predicted after a small zoom in or out (±10%) or shift (±5% diagonally), the same kinds of changes as the training augmentation, with white fill. Averaging the probabilities of the 5 views smooths out predictions that depend on the exact position or size of the emoji. The predictions of every view are saved, so the gain of TTA is measured on the out-of-fold images (see "Evaluation").

In [ ]:
# (zoom, shift as a fraction of the image size). The first view is the image itself
TTA_VIEWS = [(1.0, 0.0), (1.1, 0.0), (0.9, 0.0), (1.0, 0.05), (1.0, -0.05)]

def zoom_and_shift(images, zoom, shift):
    # Each output pixel p reads the input pixel at center + (p - center) / zoom - shift * IMG_SIZE (on both axes)
    center = (IMG_SIZE - 1) / 2
    offset = center * (1 - 1 / zoom) - shift * IMG_SIZE
    transform = np.tile(np.array([1 / zoom, 0, offset, 0, 1 / zoom, offset, 0, 0], dtype="float32"), (len(images), 1))
    return ops.image.affine_transform(images.astype("float32"), transform, fill_value=255.0)

def predict_views(model, X, chunk_size=1024):
    """Predicted probabilities for each TTA view, shape (views, images, classes)"""
    probs = np.zeros((len(TTA_VIEWS), len(X), num_classes), dtype="float32")
    for start in range(0, len(X), chunk_size):
        chunk = X[start:start + chunk_size]
        for v, (zoom, shift) in enumerate(TTA_VIEWS):
            view = chunk if v == 0 else zoom_and_shift(chunk, zoom, shift)
            probs[v, start:start + len(chunk)] = model.predict(view, batch_size=BATCH_SIZE, verbose=0)
    return probs

# Check the views on one image
fig, axes = plt.subplots(1, len(TTA_VIEWS), figsize=(2.5 * len(TTA_VIEWS), 2.8))
for ax, (zoom, shift) in zip(axes, TTA_VIEWS):
    ax.imshow(ops.convert_to_numpy(zoom_and_shift(X_train[:1], zoom, shift))[0].astype("uint8"))
    ax.set_title(f"zoom {zoom}, shift {shift}", fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
# Out-of-fold predictions: each training image is predicted by the model that did not train on it
oof_views = np.zeros((len(TTA_VIEWS), len(X_train), num_classes), dtype="float32")
test_views = {}
histories = {}

# The ensemble checks that all runs used the same folds
fold_file = RUN_DIR / "fold_of.npy"
if fold_file.exists():
    assert np.array_equal(np.load(fold_file), fold_of), f"{RUN_DIR} was trained with other folds"
np.save(fold_file, fold_of)

for fold in FOLDS_TO_RUN:
    is_val = fold_of == fold
    oof_file = RUN_DIR / f"fold{fold}_oof_views.npy"
    test_file = RUN_DIR / f"fold{fold}_test_views.npy"
    # Written last: if it exists, the fold finished and its predictions are complete
    log_file = RUN_DIR / f"fold{fold}_log.json"

    if log_file.exists():
        print(f"Fold {fold + 1}: already trained, loading its predictions")
    else:
        print(f"===== Fold {fold + 1}/{N_FOLDS} =====")
        start = time.time()
        model, history = train_fold(fold)
        np.save(oof_file, predict_views(model, X_train[is_val]))
        np.save(test_file, predict_views(model, X_test))
        log_file.write_text(json.dumps({"minutes": (time.time() - start) / 60, "history": history}))

    log = json.loads(log_file.read_text())
    histories[fold] = log["history"]
    oof_views[:, is_val] = np.load(oof_file)
    test_views[fold] = np.load(test_file)

    fold_acc = np.mean(oof_views[0, is_val].argmax(axis=1) == y_train_int[is_val])
    fold_acc_tta = np.mean(oof_views[:, is_val].mean(axis=0).argmax(axis=1) == y_train_int[is_val])
    print(f"Fold {fold + 1} validation accuracy: {fold_acc:.4f}, with TTA: {fold_acc_tta:.4f} "
          f"(trained in {log['minutes']:.0f} min)")

In [ ]:
fig, axes = plt.subplots(1, len(FOLDS_TO_RUN), figsize=(4 * len(FOLDS_TO_RUN), 3.5), sharey=True, squeeze=False)
for ax, fold in zip(axes[0], FOLDS_TO_RUN):
    pd.DataFrame(histories[fold])[["accuracy", "val_accuracy"]].plot(ax=ax, grid=True)
    ax.axvline(HEAD_EPOCHS - 0.5, color="gray", linestyle="--")  # start of fine-tuning
    ax.set_title(f"Fold {fold + 1}")
    ax.set_xlabel("Epoch")
plt.show()

### Evaluation

The cross-validation accuracy is the mean of the 5 fold accuracies; the spread across folds shows how much the score depends on the split. It is given without and with TTA; TTA is used for the rest of the notebook (and for this run in the ensemble) if it improves the out-of-fold accuracy. The confusion matrix uses the out-of-fold predictions, i.e. all 9,879 training images, each predicted by the model that did not train on it.

In [ ]:
def accuracy_and_log_loss(probs, y_int):
    # Plain log loss, without the label smoothing of the training loss
    p_true = probs[np.arange(len(y_int)), y_int]
    return np.mean(probs.argmax(axis=1) == y_int), -np.mean(np.log(np.clip(p_true, 1e-7, 1)))

fold_scores = {}
for fold in FOLDS_TO_RUN:
    is_val = fold_of == fold
    acc, ll = accuracy_and_log_loss(oof_views[0, is_val], y_train_int[is_val])
    acc_tta, ll_tta = accuracy_and_log_loss(oof_views[:, is_val].mean(axis=0), y_train_int[is_val])
    fold_scores[fold + 1] = {"accuracy": acc, "accuracy (TTA)": acc_tta, "log loss": ll, "log loss (TTA)": ll_tta}
fold_scores = pd.DataFrame(fold_scores).T.rename_axis("fold")
display(fold_scores.round(4))
for column in ["accuracy", "accuracy (TTA)"]:
    spread = f" ± {fold_scores[column].std():.4f}" if len(fold_scores) > 1 else " (a single fold)"
    print(f"Cross-validation {column}: {fold_scores[column].mean():.4f}{spread}")

was_validated = np.isin(fold_of, FOLDS_TO_RUN)
USE_TTA = bool(fold_scores["accuracy (TTA)"].mean() >= fold_scores["accuracy"].mean())
oof_probs = oof_views.mean(axis=0) if USE_TTA else oof_views[0]
print(f"Using TTA from here on: {USE_TTA}")

In [ ]:
# Per-vendor results on the out-of-fold predictions: rows are true vendors, columns are predictions
val_true = y_train[was_validated]
val_pred = classes[oof_probs[was_validated].argmax(axis=1)]

confusion = pd.crosstab(val_true, val_pred, rownames=["true"], colnames=["predicted"])
confusion = confusion.reindex(index=classes, columns=classes, fill_value=0)
display(confusion)

# Recall: share of each vendor's images that were predicted correctly
# Precision: share of the predictions of a vendor that are correct. A low precision reveals a "sink" class,
# predicted whenever the model hesitates, which recall alone hides
correct = np.diag(confusion)
per_vendor = pd.DataFrame({
    "images": confusion.sum(axis=1),
    "recall": correct / confusion.sum(axis=1),
    "precision": correct / confusion.sum(axis=0).replace(0, np.nan),
})
per_vendor["f1"] = (2 * per_vendor.precision * per_vendor.recall / (per_vendor.precision + per_vendor.recall)).fillna(0)
display(per_vendor.sort_values("f1").round(3))
print(f"Macro-F1: {per_vendor.f1.mean():.4f}")

In [ ]:
# Main confusions, both directions counted together. For comparison, the 128 px model of main-finetuning.ipynb
# made about 966 errors, including Facebook ↔ WhatsApp 299, Apple ↔ Samsung 158, Google ↔ Samsung 121,
# Apple ↔ Google 88 and Messenger ↔ Mozilla 54. Higher resolution should mostly reduce Facebook ↔ WhatsApp
errors = confusion.to_numpy().copy()
np.fill_diagonal(errors, 0)
pairs = pd.Series({
    f"{classes[i]} ↔ {classes[j]}": errors[i, j] + errors[j, i]
    for i in range(num_classes) for j in range(i + 1, num_classes)
}).sort_values(ascending=False)
display(pd.DataFrame({"errors": pairs, "share of errors": pairs / errors.sum()}).head(8).round(3))
print(f"Total errors: {errors.sum()}")

### Diagnostics

Kaggle scores accuracy, but accuracy alone does not say why the model fails, and it is noisy (on one fold, 1 image = 0.05%). On the out-of-fold predictions:
- **Top-2 accuracy**: how often the true vendor is the model's first or second choice.
- **Log loss**: how much probability the model gives to the true vendor. Less noisy than accuracy for comparing two runs that use the same label smoothing, and the score to weight models in an ensemble.
- **Calibration**: whether a prediction made with 80% confidence is right about 80% of the time. Matters before trusting the confidence, e.g. to select test images for pseudo-labelling.
- **Misclassified images**: what the errors look like (the same emoji drawn by several vendors, blurry enlarged images, mislabelled images...).
- **Errors by source group** (original size + PNG mode, analysis only): how many errors a model given the metadata would have avoided. Tests whether the remaining gap to 0.99 is a dataset shortcut rather than a modelling failure.

In [ ]:
val_probs = oof_probs[was_validated]
val_true_int = y_train_int[was_validated]

# Top-2 accuracy: the true vendor is the model's first or second choice. Much higher than top-1 means the model
# "almost knows" (ensembles and TTA can help); close to top-1 means it is missing information
top2 = np.argsort(val_probs, axis=1)[:, -2:]
top2_acc = np.mean((top2 == val_true_int[:, None]).any(axis=1))

# Log loss: average of -log(probability given to the true vendor). Plain log loss, without the label smoothing
# of the training loss. Uses the full probabilities, so it is less noisy than accuracy for comparing runs
p_true = val_probs[np.arange(len(val_true_int)), val_true_int]
log_loss = -np.mean(np.log(np.clip(p_true, 1e-7, 1)))

print(f"Top-1 accuracy: {np.mean(val_probs.argmax(axis=1) == val_true_int):.4f}")
print(f"Top-2 accuracy: {top2_acc:.4f}")
print(f"Log loss:       {log_loss:.4f}")

In [ ]:
# Calibration: within each confidence range, how often is the prediction right?
# For a calibrated model, accuracy matches the mean confidence in each row. Label smoothing (0.1) keeps the
# confidence below about 0.91, so the top row stays (almost) empty
confidence = val_probs.max(axis=1)
is_correct = val_probs.argmax(axis=1) == val_true_int
calibration = (
    pd.DataFrame({"confidence": confidence, "correct": is_correct})
    .groupby(pd.cut(confidence, np.linspace(0.1, 1.0, 10)), observed=True)
    .agg(images=("correct", "size"), mean_confidence=("confidence", "mean"), accuracy=("correct", "mean"))
    .rename_axis("confidence range")
)
display(calibration.round(3))

In [ ]:
# Misclassified validation images. "random" shows typical errors; "most_confident" shows the errors the model
# is surest about, which is where mislabelled training images would appear
ERRORS_SHOWN = 32
ERROR_ORDER = "random"  # or "most_confident"

val_idx = np.flatnonzero(was_validated)
error_idx = val_idx[~is_correct]
if ERROR_ORDER == "random":
    shown = np.random.default_rng(0).choice(error_idx, size=min(ERRORS_SHOWN, len(error_idx)), replace=False)
else:
    shown = error_idx[np.argsort(-oof_probs[error_idx].max(axis=1))][:ERRORS_SHOWN]

fig, axes = plt.subplots(4, 8, figsize=(16, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, shown):
    ax.imshow(X_train[i])
    ax.set_title(f"{y_train[i]}\n→ {classes[oof_probs[i].argmax()]} ({oof_probs[i].max():.2f})", fontsize=9)
fig.suptitle("Misclassified images: true vendor → predicted vendor (confidence)")
plt.tight_layout()
plt.show()

In [ ]:
# Errors by source group (original size + PNG mode). Analysis only: the model never sees these.
# An error is "metadata-avoidable" when the predicted vendor almost never occurs in the image's source group
# (< 5% of its images): a model given the size and mode would have ruled it out
group_share = pd.crosstab(X_train_sources, y_train, normalize="index").reindex(columns=classes, fill_value=0)
val_sources = X_train_sources[was_validated]
pred_share = group_share.to_numpy()[group_share.index.get_indexer(val_sources), val_probs.argmax(axis=1)]
avoidable = ~is_correct & (pred_share < 0.05)

by_source = (
    pd.DataFrame({"source": val_sources, "vendor": val_true, "correct": is_correct, "avoidable": avoidable})
    .groupby("source")
    .agg(
        images=("correct", "size"),
        vendors=("vendor", lambda v: ", ".join(f"{name} {n}" for name, n in v.value_counts().items())),
        accuracy=("correct", "mean"),
        errors=("correct", lambda c: (~c).sum()),
        metadata_avoidable=("avoidable", "sum"),
    )
    .sort_values("images", ascending=False)
)
display(by_source.round(3))
n_errors = (~is_correct).sum()
print(f"Metadata-avoidable errors: {avoidable.sum()} of {n_errors} ({avoidable.sum() / max(n_errors, 1):.0%})")

## Ensemble of runs

Each run (backbone + image size) saved its out-of-fold and test predictions in its own folder of `OUTPUT_DIR`. All runs use the same folds, so their out-of-fold predictions can be averaged image by image, and the ensemble is scored exactly like a single model: on images that none of its models trained on. Different networks make different mistakes, so their average is usually more accurate than the best of them. With a single run, this section simply produces that run's submission.

Then a **per-vendor bias correction**: adding a constant to each vendor's log-probability shifts the decision boundaries, for example away from a "sink" vendor that is predicted whenever the model hesitates (Samsung for the 128 px model). The 7 constants are chosen to maximize the out-of-fold accuracy. To check that this does not overfit, they are also fitted on all folds but one and scored on the remaining fold, and they are only used if that improves the accuracy.

In [ ]:
# None: every run found in OUTPUT_DIR. Or a list of run folders, e.g. ["efficientnetv2b0_224", "convnext_tiny_224"]
ENSEMBLE_RUNS = None

def load_run(run_dir):
    """Finished folds, out-of-fold probabilities (NaN on unfinished folds), mean test probabilities of a run,
    and whether TTA is used for it (if it improves the run's out-of-fold accuracy)"""
    assert np.array_equal(np.load(run_dir / "fold_of.npy"), fold_of), f"{run_dir.name} was trained with other folds"
    folds = [f for f in range(N_FOLDS) if (run_dir / f"fold{f}_log.json").exists()]
    oof_views = np.full((len(TTA_VIEWS), len(X_train), num_classes), np.nan)
    for f in folds:
        oof_views[:, fold_of == f] = np.load(run_dir / f"fold{f}_oof_views.npy")
    test_views = np.mean([np.load(run_dir / f"fold{f}_test_views.npy") for f in folds], axis=0)
    done = np.isin(fold_of, folds)
    acc = np.mean(oof_views[0, done].argmax(axis=1) == y_train_int[done])
    acc_tta = np.mean(oof_views[:, done].mean(axis=0).argmax(axis=1) == y_train_int[done])
    if acc_tta >= acc:
        return folds, oof_views.mean(axis=0), test_views.mean(axis=0), True
    return folds, oof_views[0], test_views[0], False

if ENSEMBLE_RUNS is None:
    run_dirs = sorted(d for d in OUTPUT_DIR.iterdir() if (d / "fold_of.npy").exists())
else:
    run_dirs = [OUTPUT_DIR / name for name in ENSEMBLE_RUNS]
runs = {d.name: load_run(d) for d in run_dirs if any((d / f"fold{f}_log.json").exists() for f in range(N_FOLDS))}

# Every run and their average are scored on the same images: those of the folds finished in all runs
common_folds = sorted(set.intersection(*(set(folds) for folds, _, _, _ in runs.values())))
common = np.isin(fold_of, common_folds)
y_common = y_train_int[common]
ens_oof = np.mean([oof[common] for _, oof, _, _ in runs.values()], axis=0)
ens_test = np.mean([test for _, _, test, _ in runs.values()], axis=0)

run_scores = pd.DataFrame(
    {name: accuracy_and_log_loss(oof[common], y_common) for name, (_, oof, _, _) in runs.items()},
    index=["accuracy", "log loss"],
).T
run_scores["TTA"] = [use_tta for _, _, _, use_tta in runs.values()]
if len(runs) > 1:
    run_scores.loc["ensemble (average)"] = [*accuracy_and_log_loss(ens_oof, y_common), ""]
display(run_scores.round(4))
print(f"Scored on {common.sum()} out-of-fold images (folds {[f + 1 for f in common_folds]})")

In [ ]:
def fit_offsets(log_probs, y_int, grid=np.linspace(-1, 1, 41), rounds=3):
    """Per-vendor constants added to the log-probabilities, tuned one vendor at a time to maximize accuracy"""
    offsets = np.zeros(num_classes)
    for _ in range(rounds):
        for c in range(num_classes):
            accs = []
            for value in grid:
                offsets[c] = value
                accs.append(np.mean((log_probs + offsets).argmax(axis=1) == y_int))
            # Among the values that reach the best accuracy, keep the smallest correction
            best = np.flatnonzero(np.array(accs) == max(accs))
            offsets[c] = grid[best[np.argmin(np.abs(grid[best]))]]
    return offsets

log_oof = np.log(np.clip(ens_oof, 1e-7, 1))
acc_before = np.mean(log_oof.argmax(axis=1) == y_common)
offsets = np.zeros(num_classes)
if len(common_folds) > 1:
    # Honest check: the offsets are fitted on the other folds and scored on the held-out fold
    fold_common = fold_of[common]
    held_out_correct = np.zeros(len(y_common), dtype=bool)
    for f in common_folds:
        held = fold_common == f
        held_offsets = fit_offsets(log_oof[~held], y_common[~held])
        held_out_correct[held] = (log_oof[held] + held_offsets).argmax(axis=1) == y_common[held]
    acc_after = held_out_correct.mean()
    print(f"Out-of-fold accuracy without correction: {acc_before:.4f}, with correction (held-out folds): {acc_after:.4f}")
    if acc_after > acc_before:
        offsets = fit_offsets(log_oof, y_common)
print(f"Bias correction used: {bool(offsets.any())}")
display(pd.Series(offsets, index=classes, name="log-probability offset").round(2).to_frame())

## Submission

The test predictions are the average of all fold models of all runs in the ensemble, with the bias correction if it was kept.

In [ ]:
y_pred = classes[(np.log(np.clip(ens_test, 1e-7, 1)) + offsets).argmax(axis=1)]

submission = pd.DataFrame({
    "Id": X_test_ids,
    "Label": y_pred
})
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print(f"Runs: {', '.join(runs)} (bias correction: {bool(offsets.any())})")
display(submission.Label.value_counts().to_frame().T)
submission.head()